In [2]:
import os
import sqlite3
import pandas as pd

# 1. Define paths
CSV_PATH = "/home/madhan/Projects/SteamBot/games.csv"
DB_PATH = "/home/madhan/Projects/SteamBot/games.db"

In [3]:
df_raw = pd.read_csv(CSV_PATH, index_col=False, low_memory=False)

# 3. Detect and fix the header offset caused by 'DiscountDLC count'
cols = list(df_raw.columns)
if "DiscountDLC count" in cols:
    idx = cols.index("DiscountDLC count")
    cols[idx] = "Discount"
    cols.insert(idx + 1, "DLC count")

    # Re-read the CSV using the properly expanded list of column names
    df = pd.read_csv(CSV_PATH, names=cols, skiprows=1, low_memory=False)
else:
    df = df_raw.copy()

In [4]:
# 4. Standardize column names (map messy headers to clean SQL column names)
# Check the exact raw column name for user score (e.g., Metacritic / user_score)
column_mapping = {
    "AppID": "appid",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Metacritic score": "user_score",  # Or 'user_score' depending on your raw CSV header
    "Estimated owners": "estimated_owners",
    "Peak CCU": "peak_ccu",
    "Required age": "required_age",
    "Developers": "developers",
    "Publishers": "publishers",
    "Categories": "categories",
    "Genres": "genres",
    "Tags": "tags",
}

# Keep only columns that exist in the DataFrame and rename them
existing_cols = {
    k: v for k, v in column_mapping.items() if k in df.columns or v in df.columns
}
df_clean = df[list(existing_cols.keys())].rename(columns=existing_cols)

In [5]:
# 5. Clean Data Types
df_clean["appid"] = pd.to_numeric(df_clean["appid"], errors="coerce")
df_clean.dropna(subset=["appid"], inplace=True)
df_clean["appid"] = df_clean["appid"].astype(int)

if "price" in df_clean.columns:
    df_clean["price"] = pd.to_numeric(df_clean["price"], errors="coerce").fillna(0.0)

In [6]:
# 6. Save directly to SQLite
conn = sqlite3.connect(DB_PATH)
df_clean.to_sql("games", conn, if_exists="replace", index=False)

# Create index on appid for fast lookups
cursor = conn.cursor()
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_appid ON games (appid);")
conn.commit()
conn.close()

print("Successfully regenerated games.db with aligned columns!")

Successfully regenerated games.db with aligned columns!


In [8]:
conn = sqlite3.connect("/home/madhan/Projects/SteamBot/games.db")
verification_df = pd.read_sql_query(
    "SELECT appid, name, price, user_score FROM games WHERE user_score IS NOT NULL LIMIT 5;",
    conn,
)
conn.close()

verification_df

,appid,name,price,user_score
0,2539430,Black Dragon Mage Playtest,0.00,0
1,496350,Supipara - Chapter 1 Spring Has Come!,5.24,0
2,1034400,Mystery Solitaire The Black Raven,4.99,0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,8.99,0
4,3631080,Maze Quest VR,4.99,0
